# testAPI — exploring the `espn-api` library

Scratch notebook for seeing what [`espn-api`](https://github.com/cwendt94/espn-api) gives us before
defining domain objects. Each section looks at one of its object types and the data it carries.

**Setup:** set `ESPN_LEAGUE_ID` (and `ESPN_S2` / `ESPN_SWID` for private leagues) in `.env`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from espn_api.football import League

from fantasy_analysis.config import settings

pd.set_option("display.max_columns", 50)


def attrs(obj) -> dict:
    """An object's attributes, with nested objects/lists shown as type names."""
    return {
        k: v if isinstance(v, (str, int, float, bool, type(None))) else f"<{type(v).__name__}>"
        for k, v in vars(obj).items()
    }


def to_df(objs) -> pd.DataFrame:
    """One row per object, scalar attributes only."""
    return pd.DataFrame([attrs(o) for o in objs])

## League

In [ ]:
league = League(
    league_id=settings.espn_league_id,
    year=settings.espn_season,
    espn_s2=settings.espn_s2,
    swid=settings.espn_swid,
)
print(league, "| current week:", league.current_week)
attrs(league)

In [ ]:
# League settings: roster slots, scoring format, playoff setup, etc.
vars(league.settings)

In [ ]:
league.members

## Teams

In [ ]:
to_df(league.teams)

In [ ]:
team = league.teams[0]
attrs(team)

In [ ]:
# Week-by-week results for one team: opponent, score, outcome
pd.DataFrame(
    {
        "opponent": [opp.team_name for opp in team.schedule],
        "score": team.scores,
        "outcome": team.outcomes,
    },
    index=pd.RangeIndex(1, len(team.schedule) + 1, name="week"),
)

In [ ]:
standings = league.standings()
pd.DataFrame(
    [(t.team_name, t.wins, t.losses, t.points_for, t.points_against) for t in standings],
    columns=["team", "wins", "losses", "points_for", "points_against"],
)

In [ ]:
league.power_rankings()

## Players (rosters)

In [ ]:
roster_df = to_df(p for t in league.teams for p in t.roster)
print(roster_df.shape)
roster_df.head()

In [ ]:
player = team.roster[0]
attrs(player)

In [ ]:
# player.stats is keyed by scoring period (0 = season total)
player.stats

## Matchups and box scores

In [ ]:
WEEK = league.current_week
to_df(league.scoreboard(week=WEEK))

In [ ]:
box_scores = league.box_scores(week=WEEK)
for box in box_scores:
    home = box.home_team.team_name
    away = getattr(box.away_team, "team_name", "BYE")
    print(f"{home} {box.home_score} - {box.away_score} {away}")

In [ ]:
# Starters and bench for one side of a matchup, with actual vs. projected points
box = box_scores[0]
to_df(box.home_lineup)

In [ ]:
# Per-stat breakdown for one player in that week
lineup_player = box.home_lineup[0]
lineup_player.name, lineup_player.points_breakdown

## Draft

In [ ]:
to_df(league.draft)

## Free agents

In [ ]:
to_df(league.free_agents(size=25))

## Transactions and activity

In [ ]:
activity = league.recent_activity(size=25)
pd.DataFrame(
    [
        (a.date, t.team_name if t else None, action, p.name if p else None, bid)
        for a in activity
        for t, action, p, bid in a.actions
    ],
    columns=["date", "team", "action", "player", "bid_amount"],
)

In [ ]:
to_df(league.transactions(scoring_period=WEEK))

## Raw access

For data the library doesn't model, `league.espn_request` calls the endpoint directly with auth handled.
Views include `mSettings`, `mTeam`, `mRoster`, `mMatchupScore`, `mDraftDetail`, `mTransactions2`.

In [ ]:
raw = league.espn_request.league_get(params={"view": "mSettings"})
raw["settings"]["scoringSettings"].keys()